# Friends' weekend — do we reply faster? (week 4, distributions)

Builds the Analysis 13 chart from `analysis-log.md`, for the week-4 distributions
assignment (`week4-distribution-requirements.md`).

**Question (Stage 1):** around the friends' weekend we have something to coordinate
("where are you?", "do you want a drink?") — do we reply to each other faster than on
an ordinary weekend?

**Primary statistic, fixed before looking:** per Fri–Sun block, the median reply gap;
the event weekend's median is at least 20% lower than ordinary Fri–Sun weekends.
Weekends with fewer than 2 reply gaps are left out.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 06-08: `timestamp` is stored as strings; the `+00:00` is only a label --
# these are the raw export's local clock times.
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)

NEW_CONVERSATION_MIN = 6 * 60  # a gap above this starts a new conversation, not a slow reply
MIN_REPLIES = 2  # a weekend needs at least this many reply gaps to count
FRIENDS_WEEKENDS = {"2024": pd.Timestamp("2024-04-19"), "2025": pd.Timestamp("2025-09-19")}

## Reply gap

A reply = the first message by a different author than the one before it (photos and
stickers count). The gap runs from the *last* message of the previous author's run.
Gaps over 6 hours start a new conversation and are not replies.

In [ ]:
minutes_since_previous = own["timestamp"].diff().dt.total_seconds() / 60
author_changed = own["author"].ne(own["author"].shift()) & minutes_since_previous.notna()

own["reply_gap_min"] = minutes_since_previous.where(author_changed)
own["new_conversation"] = own["reply_gap_min"] > NEW_CONVERSATION_MIN

replies = own[author_changed & ~own["new_conversation"]].copy()
# a Fri-Sun block is named after its Friday; a reply belongs to the day it was sent
replies = replies[replies["timestamp"].dt.weekday >= 4]
days_since_friday = pd.to_timedelta(replies["timestamp"].dt.weekday - 4, unit="D")
replies["weekend"] = replies["timestamp"].dt.normalize() - days_since_friday
replies["event"] = replies["weekend"].map({d: y for y, d in FRIENDS_WEEKENDS.items()}).fillna("none")

print(f"{len(replies)} Fri-Sun reply gaps over {replies['weekend'].nunique()} weekends")

## Stage 3 — shape (ordinary weekends only; event weekends not shown yet)

Two things to look at: the reply gaps themselves, and the per-weekend median that the
claim is about.

In [ ]:
ordinary = replies[replies["event"] == "none"]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(ordinary["reply_gap_min"], bins=range(0, NEW_CONVERSATION_MIN + 5, 5), color="grey")
ax.set_xlabel("reply gap (minutes, 5-minute bins)")
ax.set_ylabel("reply gaps")
ax.set_title("Reply gaps on ordinary Fri-Sun weekends")
plt.show()

print("share of reply gaps of exactly 0 minutes:", round((ordinary["reply_gap_min"] == 0).mean(), 2))
ordinary["reply_gap_min"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1)

In [ ]:
per_weekend = ordinary.groupby("weekend")["reply_gap_min"].agg(median="median", replies="size")
counted = per_weekend[per_weekend["replies"] >= MIN_REPLIES]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(counted["median"], bins=range(0, int(counted["median"].max()) + 2, 1), color="grey")
axes[0].set_xlabel("median reply gap of the weekend (minutes)")
axes[0].set_ylabel("weekends")
axes[0].set_title(f"{len(counted)} ordinary weekends with ≥ {MIN_REPLIES} reply gaps")
axes[1].hist(per_weekend["replies"], bins=30, color="grey")
axes[1].set_xlabel("reply gaps in the weekend")
axes[1].set_ylabel("weekends")
axes[1].set_title("How many reply gaps a weekend has")
plt.tight_layout()
plt.show()

print(f"weekends left out (< {MIN_REPLIES} reply gaps): {(per_weekend['replies'] < MIN_REPLIES).sum()}")
counted["median"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1)

## Comparable weekends and the rank test (Stage 3 decisions)

- Comparable = ordinary Fri–Sun with **≥ 50 reply gaps** (as busy as the friends' weekends get).
- A weekend is dropped from the comparison if a birthday falls on its Fri–Sun, or it is the
  Groningen weekend (2023-06-23..25).
- Test: **both** friends' weekends have a lower median than **80%** of comparable weekends.
  A tie counts *against* the claim.

In [ ]:
MIN_BUSY = 50
SHARE_TO_BEAT = 0.80
# day/month only, against the anonymised aliases (supplied by the student)
BIRTHDAYS = {
    "fluffy-beaver": "04-13", "effervescent-penguin": "09-16", "striking-rail": "09-22",
    "hypnotic-rabbit": "10-19", "pliable-tiger": "06-08", "rib-tickling-curlew": "08-03",
    "vibrant-barracuda": "03-10", "animated-elk": "01-12", "humorous-stingray": "07-18",
}
GRONINGEN_WEEKEND = pd.Timestamp("2023-06-23")

years = range(replies["timestamp"].dt.year.min(), replies["timestamp"].dt.year.max() + 1)
birthday_dates = pd.to_datetime([f"{y}-{md}" for y in years for md in BIRTHDAYS.values()])
birthday_weekends = {d.normalize() - pd.Timedelta(days=d.weekday() - 4) for d in birthday_dates if d.weekday() >= 4}
excluded = birthday_weekends | {GRONINGEN_WEEKEND}

weekends = replies.groupby("weekend").agg(
    median=("reply_gap_min", "median"), replies=("reply_gap_min", "size"), event=("event", "first")
)
weekends["role"] = "ordinary"
weekends.loc[weekends.index.isin(excluded), "role"] = "excluded"
weekends.loc[(weekends["role"] == "ordinary") & (weekends["replies"] >= MIN_BUSY), "role"] = "comparable"
weekends.loc[weekends["event"] != "none", "role"] = "friends' weekend"
weekends = weekends[weekends["replies"] >= MIN_REPLIES]

comparable = weekends[weekends["role"] == "comparable"]
friends = weekends[weekends["role"] == "friends' weekend"]
print(f"comparable weekends: {len(comparable)} (dropped for birthday/Groningen: "
      f"{((weekends['role'] == 'excluded') & (weekends['replies'] >= MIN_BUSY)).sum()})")

rank_test = pd.DataFrame({
    "median_min": friends["median"].to_numpy(),
    "replies": friends["replies"].to_numpy(),
    # strict: a comparable weekend with the same median counts as 'not beaten'
    "share_beaten": [(comparable["median"] > m).mean() for m in friends["median"]],
}, index=friends["event"])
rank_test["passes"] = rank_test["share_beaten"] >= SHARE_TO_BEAT
rank_test

## Stage 4 — sketches (scratch, not the final chart)

A = distributions (ECDF of all reply gaps, the student's preference) ·
B = distributions (the test itself: one dot per weekend) ·
C = time (the stretch: median per weekend across the export).

In [ ]:
FRIENDS_COLORS = {"2024": "#d95f02", "2025": "#1b9e77"}
comparable_gaps = replies[replies["weekend"].isin(comparable.index)]["reply_gap_min"]

def ecdf(values):
    x = values.sort_values().to_numpy()
    return x, (pd.Series(range(1, len(x) + 1)) / len(x)).to_numpy()

fig, ax = plt.subplots(figsize=(9, 5))
for weekend, gaps in replies[replies["weekend"].isin(comparable.index)].groupby("weekend")["reply_gap_min"]:
    ax.step(*ecdf(gaps), where="post", color="lightgrey", lw=0.8)
ax.step(*ecdf(comparable_gaps), where="post", color="black", lw=2, label=f"{len(comparable)} comparable weekends (pooled)")
for year, colour in FRIENDS_COLORS.items():
    gaps = replies[replies["event"] == year]["reply_gap_min"]
    ax.step(*ecdf(gaps), where="post", color=colour, lw=2.5, label=f"friends' weekend {year}")
ax.axhline(0.5, color="grey", ls=":", lw=1)
ax.set_xlim(0, 60)
ax.set_xlabel("reply gap (minutes) — axis cut at 60; all lines reach 100% at 360")
ax.set_ylabel("share of replies within x minutes")
ax.set_title("Sketch A — cumulative share of replies, friends' weekends vs comparable weekends\n(thin grey = each comparable weekend)")
ax.legend(loc="lower right")
fig.savefig("sketch-A-reply-gap-ecdf.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.scatter(comparable["median"], [0] * len(comparable), color="black", alpha=0.6, s=40, label="comparable weekend")
for year, colour in FRIENDS_COLORS.items():
    m = friends.loc[friends["event"] == year, "median"]
    ax.scatter(m, [0], color=colour, s=160, marker="D", label=f"friends' weekend {year}", zorder=3)
cut = comparable["median"].quantile(1 - SHARE_TO_BEAT)
ax.axvline(cut, color="grey", ls="--", lw=1)
ax.text(cut, 0.35, f" faster than 80% of comparable weekends ←", ha="right", va="center", color="grey", fontsize=9)
ax.set_yticks([])
ax.set_ylim(-0.6, 0.6)
ax.set_xlabel("median reply gap of the weekend (minutes)")
ax.set_title("Sketch B — where the friends' weekends rank among comparable weekends")
ax.legend(loc="upper right", fontsize=8, ncol=3, bbox_to_anchor=(1, -0.3))
fig.savefig("sketch-B-reply-gap-rank.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
other = weekends[weekends["role"].isin(["ordinary", "excluded"])]
ax.scatter(other.index, other["median"], color="lightgrey", s=12, label="quieter / excluded weekends")
ax.scatter(comparable.index, comparable["median"], color="black", s=25, label="comparable (≥ 50 reply gaps)")
for year, colour in FRIENDS_COLORS.items():
    row = friends[friends["event"] == year]
    ax.scatter(row.index, row["median"], color=colour, s=140, marker="D", label=f"friends' weekend {year}", zorder=3)
ax.set_yscale("symlog", linthresh=10)
ax.set_ylabel("median reply gap (minutes, symlog)")
ax.set_title("Sketch C — median reply gap per Fri–Sun weekend, 2020–2026")
ax.legend(fontsize=8, loc="upper left")
fig.savefig("sketch-C-reply-gap-over-time.png", dpi=120, bbox_inches="tight")
plt.show()

## The chart — probability per minute (Stage 4 choice)

Reply gaps are whole minutes, so this is a probability mass function, not a smooth density.
One bar per minute for 0–10; wider bins further out, with the height divided by the bin width
so every bar reads as *share of replies per minute*. Comparable weekends = the average of their
per-weekend distributions (each weekend one vote).

In [ ]:
EDGES = list(range(0, 11)) + [11, 21, 61, NEW_CONVERSATION_MIN + 1]
LABELS = [str(m) for m in range(0, 11)] + ["11–20", "21–60", "61–360"]
WIDTHS = pd.Series([hi - lo for lo, hi in zip(EDGES[:-1], EDGES[1:])], index=LABELS)

def per_minute(gaps: pd.Series) -> pd.Series:
    share = pd.cut(gaps, EDGES, right=False, labels=LABELS).value_counts(normalize=True, sort=False)
    return share / WIDTHS

comparable_pmf = (
    replies[replies["weekend"].isin(comparable.index)]
    .groupby("weekend")["reply_gap_min"].apply(per_minute).unstack().mean()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
x = range(len(LABELS))
for ax, (year, colour) in zip(axes, FRIENDS_COLORS.items()):
    friends_pmf = per_minute(replies[replies["event"] == year]["reply_gap_min"])
    ax.bar(x, comparable_pmf, width=0.8, color="lightgrey", label=f"{len(comparable)} comparable busy weekends")
    ax.bar(x, friends_pmf, width=0.35, color=colour, label=f"friends' weekend {year}")
    ax.set_xticks(list(x), LABELS, rotation=45)
    ax.set_xlabel("reply gap (minutes)")
    ax.set_title(f"Friends' weekend {year}")
    ax.legend(frameon=False)
axes[0].set_ylabel("share of replies per minute")
fig.suptitle("Reply gaps on the friends' weekend vs equally busy weekends")
plt.tight_layout()
fig.savefig("draft-reply-gap-pmf.png", dpi=120, bbox_inches="tight")
plt.show()

## Final chart — after the Stage 5 critique

One graph: both friends' weekends against the comparable weekends. The grey whisker is the
range the middle 80% of comparable weekends fall in (10th–90th percentile, matching the 80%
test), so the null — *the friends' weekends sit inside ordinary variation* — is visible.

In [ ]:
per_comparable = (
    replies[replies["weekend"].isin(comparable.index)]
    .groupby("weekend")["reply_gap_min"].apply(per_minute).unstack()
)
low, high = per_comparable.quantile(0.10), per_comparable.quantile(0.90)

fig, ax = plt.subplots(figsize=(11, 5))
x = pd.Series(range(len(LABELS)), index=LABELS)
ax.bar(x, comparable_pmf, width=0.75, color="#e3e3e3", label=f"{len(comparable)} equally busy weekends (average)")
ax.vlines(x, low, high, color="#9a9a9a", lw=2, label="middle 80% of those weekends")
for offset, (year, colour) in zip([-0.15, 0.15], FRIENDS_COLORS.items()):
    friends_pmf = per_minute(replies[replies["event"] == year]["reply_gap_min"])
    ax.scatter(x + offset, friends_pmf, color=colour, s=55, zorder=3, label=f"friends' weekend {year}")
ax.axvline(10.5, color="grey", lw=0.8, ls=":")
ax.text(10.6, ax.get_ylim()[1] * 0.95, "wider bins →\n(height per minute)", fontsize=8, color="grey", va="top")
ax.set_xticks(x, LABELS)
ax.set_xlabel("reply gap (minutes)")
ax.set_ylabel("share of replies per minute")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("On the friends' weekend we don't reply faster than on any other busy weekend", loc="left")
ax.legend(frameon=False)
fig.savefig("friends-weekend-reply-gap-final.png", dpi=120, bbox_inches="tight")
plt.show()

## Stage 6 — sensitivity: the busy line at ≥ 30 and ≥ 75 (≥ 50 stays primary)

In [ ]:
def rank_for(min_busy: int) -> pd.DataFrame:
    pool = weekends[(weekends["role"].isin(["comparable", "ordinary"])) & (weekends["replies"] >= min_busy)]
    return pd.DataFrame({
        "busy_line": f">= {min_busy}",
        "comparable_weekends": len(pool),
        "share_beaten": [(pool["median"] > m).mean().round(2) for m in friends["median"]],
    }, index=friends["event"])

pd.concat([rank_for(n) for n in (30, MIN_BUSY, 75)])